In [5]:
# ============================================================
# PropTech AVM - Modeling Team
# Stacking Ensemble:
# XGBoost + LightGBM + Random Forest + Ridge Meta-Learner
#
# CSCI323 Modern Artificial Intelligence
# ============================================================

import os
import joblib
import warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import (
    mean_absolute_percentage_error,
    mean_squared_error,
    r2_score
)
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
N_SPLITS = 5

# ============================================================
# DIRECTORIES
# ============================================================

os.makedirs("models", exist_ok=True)
os.makedirs("results", exist_ok=True)

# ============================================================
# LOAD TRAIN DATA
# ============================================================

print("=" * 60)
print("Loading training data...")
print("=" * 60)

train = pd.read_csv("/content/train.csv")

print(f"Train Shape: {train.shape}")

y = train["trans_value"]

X = train.drop(
    columns=[
        "trans_value",
        "transaction_num"
    ],
    errors="ignore"
)

print(f"Feature Count: {X.shape[1]}")

# ============================================================
# TIME SERIES CV
# ============================================================

tscv = TimeSeriesSplit(
    n_splits=N_SPLITS
)

# ============================================================
# EVALUATION FUNCTION
# ============================================================

def evaluate_cv(model, X, y, cv):

    mape_scores = []
    rmse_scores = []
    r2_scores = []

    for fold, (train_idx, val_idx) in enumerate(cv.split(X), start=1):

        X_train = X.iloc[train_idx]
        X_val = X.iloc[val_idx]

        y_train = y.iloc[train_idx]
        y_val = y.iloc[val_idx]

        model.fit(X_train, y_train)

        preds = model.predict(X_val)

        mape = mean_absolute_percentage_error(
            y_val,
            preds
        )

        rmse = np.sqrt(
            mean_squared_error(
                y_val,
                preds
            )
        )

        r2 = r2_score(
            y_val,
            preds
        )

        mape_scores.append(mape)
        rmse_scores.append(rmse)
        r2_scores.append(r2)

        print(
            f"Fold {fold}: "
            f"MAPE={mape:.4f} "
            f"RMSE={rmse:.2f} "
            f"R2={r2:.4f}"
        )

    return {
        "MAPE": np.mean(mape_scores),
        "RMSE": np.mean(rmse_scores),
        "R2": np.mean(r2_scores)
    }

# ============================================================
# BASELINE XGBOOST
# ============================================================

print("\nRunning Baseline XGBoost...")

baseline_xgb = XGBRegressor(
    objective="reg:squarederror",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

baseline_results = evaluate_cv(
    baseline_xgb,
    X,
    y,
    tscv
)

print("\nBaseline Results")
print(baseline_results)

# ============================================================
# TUNING SUBSET
# ============================================================

print("\nPreparing tuning subset...")

TUNING_ROWS = min(
    150000,
    len(X)
)

X_tune = X.iloc[:TUNING_ROWS]
y_tune = y.iloc[:TUNING_ROWS]

print(f"Tuning rows: {len(X_tune)}")

# ============================================================
# XGBOOST TUNING
# ============================================================

print("\nTuning XGBoost...")

xgb_params = {
    "n_estimators": [300, 500],
    "max_depth": [6, 8, 10],
    "learning_rate": [0.03, 0.05, 0.1],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0]
}

xgb_search = RandomizedSearchCV(
    estimator=XGBRegressor(
        objective="reg:squarederror",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    param_distributions=xgb_params,
    n_iter=10,
    cv=tscv,
    scoring="neg_mean_absolute_percentage_error",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)

xgb_search.fit(
    X_tune,
    y_tune
)

best_xgb = xgb_search.best_estimator_

print("Best XGB Params:")
print(xgb_search.best_params_)

# ============================================================
# LIGHTGBM TUNING
# ============================================================

print("\nTuning LightGBM...")

lgbm_params = {
    "n_estimators": [300, 500],
    "learning_rate": [0.03, 0.05],
    "num_leaves": [31, 63, 127],
    "max_depth": [6, 8, -1]
}

lgbm_search = RandomizedSearchCV(
    estimator=LGBMRegressor(
        random_state=RANDOM_STATE
    ),
    param_distributions=lgbm_params,
    n_iter=10,
    cv=tscv,
    scoring="neg_mean_absolute_percentage_error",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)

lgbm_search.fit(
    X_tune,
    y_tune
)

best_lgbm = lgbm_search.best_estimator_

print("Best LGBM Params:")
print(lgbm_search.best_params_)

# ============================================================
# RANDOM FOREST TUNING
# ============================================================

print("\nTuning Random Forest...")

rf_params = {
    "n_estimators": [200, 300],
    "max_depth": [20, 30, None],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2]
}

rf_search = RandomizedSearchCV(
    estimator=RandomForestRegressor(
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    param_distributions=rf_params,
    n_iter=8,
    cv=tscv,
    scoring="neg_mean_absolute_percentage_error",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)

rf_search.fit(
    X_tune,
    y_tune
)

best_rf = rf_search.best_estimator_

print("Best RF Params:")
print(rf_search.best_params_)

# ============================================================
# OOF PREDICTIONS
# ============================================================

print("\nGenerating OOF predictions...")

oof_xgb = np.full(len(X), np.nan)
oof_lgbm = np.full(len(X), np.nan)
oof_rf = np.full(len(X), np.nan)

for fold, (train_idx, val_idx) in enumerate(
    tscv.split(X),
    start=1
):

    print(f"OOF Fold {fold}")

    X_train = X.iloc[train_idx]
    X_val = X.iloc[val_idx]

    y_train = y.iloc[train_idx]

    best_xgb.fit(X_train, y_train)
    best_lgbm.fit(X_train, y_train)
    best_rf.fit(X_train, y_train)

    oof_xgb[val_idx] = best_xgb.predict(X_val)
    oof_lgbm[val_idx] = best_lgbm.predict(X_val)
    oof_rf[val_idx] = best_rf.predict(X_val)

# ============================================================
# META FEATURES
# ============================================================

valid_mask = ~np.isnan(oof_xgb)

meta_X = np.column_stack([
    oof_xgb[valid_mask],
    oof_lgbm[valid_mask],
    oof_rf[valid_mask]
])

meta_y = y.iloc[valid_mask]

scaler = StandardScaler()

meta_X_scaled = scaler.fit_transform(
    meta_X
)

ridge = Ridge(alpha=1.0)

ridge.fit(
    meta_X_scaled,
    meta_y
)

# ============================================================
# RETRAIN BASE MODELS ON FULL TRAIN
# ============================================================

print("\nTraining final base learners...")

best_xgb.fit(X, y)
best_lgbm.fit(X, y)
best_rf.fit(X, y)

# ============================================================
# SAVE MODELS
# ============================================================

joblib.dump(
    best_xgb,
    "models/xgb.pkl"
)

joblib.dump(
    best_lgbm,
    "models/lgbm.pkl"
)

joblib.dump(
    best_rf,
    "models/rf.pkl"
)

joblib.dump(
    ridge,
    "models/ridge.pkl"
)

joblib.dump(
    scaler,
    "models/meta_scaler.pkl"
)

print("Models saved.")

# ============================================================
# FINAL TEST EVALUATION
# ============================================================

print("\nLoading test set...")

test = pd.read_csv("/content/test.csv")

y_test = test["trans_value"]

X_test = test.drop(
    columns=[
        "trans_value",
        "transaction_num"
    ],
    errors="ignore"
)

# ============================================================
# STACKING PREDICTIONS
# ============================================================

xgb_preds = best_xgb.predict(X_test)
lgbm_preds = best_lgbm.predict(X_test)
rf_preds = best_rf.predict(X_test)

meta_test = np.column_stack([
    xgb_preds,
    lgbm_preds,
    rf_preds
])

meta_test = scaler.transform(
    meta_test
)

final_preds = ridge.predict(
    meta_test
)

# ============================================================
# OVERALL METRICS
# ============================================================

overall_mape = mean_absolute_percentage_error(
    y_test,
    final_preds
)

overall_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        final_preds
    )
)

overall_r2 = r2_score(
    y_test,
    final_preds
)

overall_results = pd.DataFrame({
    "Metric": ["MAPE", "RMSE", "R2"],
    "Value": [
        overall_mape,
        overall_rmse,
        overall_r2
    ]
})

overall_results.to_csv(
    "results/overall_metrics.csv",
    index=False
)

print("\nFINAL RESULTS")
print(overall_results)

# ============================================================
# PROPERTY TYPE METRICS
# ============================================================

segments = {
    "Land": test["prop_type_Land"] == 1,
    "Unit": test["prop_type_Unit"] == 1,
    "Villa": test["prop_type_Villa"] == 1
}

segment_results = []

for name, mask in segments.items():

    if mask.sum() == 0:
        continue

    seg_y = y_test[mask]
    seg_pred = final_preds[mask]

    segment_results.append({
        "Property_Type": name,
        "MAPE": mean_absolute_percentage_error(
            seg_y,
            seg_pred
        ),
        "RMSE": np.sqrt(
            mean_squared_error(
                seg_y,
                seg_pred
            )
        ),
        "R2": r2_score(
            seg_y,
            seg_pred
        )
    })

segment_df = pd.DataFrame(
    segment_results
)

segment_df.to_csv(
    "results/property_type_metrics.csv",
    index=False
)

print("\nProperty Type Metrics")
print(segment_df)

print("\nProject Complete.")

Loading training data...
Train Shape: (448395, 32)
Feature Count: 30

Running Baseline XGBoost...
Fold 1: MAPE=0.2128 RMSE=1113011.55 R2=0.8131
Fold 2: MAPE=0.2132 RMSE=1064145.46 R2=0.8217
Fold 3: MAPE=0.1727 RMSE=841992.77 R2=0.9009
Fold 4: MAPE=0.1733 RMSE=872410.98 R2=0.9177
Fold 5: MAPE=0.1748 RMSE=754585.32 R2=0.9047

Baseline Results
{'MAPE': np.float64(0.18934188626870946), 'RMSE': np.float64(929229.2145827148), 'R2': np.float64(0.871603806239774)}

Preparing tuning subset...
Tuning rows: 150000

Tuning XGBoost...
Fitting 5 folds for each of 10 candidates, totalling 50 fits
Best XGB Params:
{'subsample': 0.8, 'n_estimators': 500, 'max_depth': 8, 'learning_rate': 0.05, 'colsample_bytree': 1.0}

Tuning LightGBM...
Fitting 5 folds for each of 10 candidates, totalling 50 fits
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.029025 seconds.
You can set `force_row_wi